In [ ]:
import pandas as pd
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# # Configuración de las rutas
PATH = Path().cwd()
PATH = PATH.parent if PATH.name == "notebooks" else PATH
DATA = 'data'
PROCESSED_DATA = 'processed/clean_data.csv'

print(f'La nueva ruta de los archivos ha sido asignado a: {PATH}')

In [ ]:
df = pd.read_csv(PATH / DATA / PROCESSED_DATA)

df.head()

In [ ]:
canton_map = {
    101: "SAN JOSE",
    102: "ESCAZU",
    103: "DESAMPARADOS",
    104: "PURISCAL",
    105: "TARRAZU",
    106: "ASERRI",
    107: "MORA",
    108: "GOICOECHEA",
    109: "SANTA ANA",
    110: "ALAJUELITA",
    111: "CORONADO",
    112: "ACOSTA",
    113: "TIBAS",
    114: "MORAVIA",
    115: "MONTES DE OCA",
    116: "TURRUBARES",
    117: "DOTA",
    118: "CURRIDABAT",
    119: "PEREZ ZELEDON",
    120: "LEON CORTES",
    201: "ALAJUELA",
    202: "SAN RAMON",
    203: "GRECIA",
    204: "SAN MATEO",
    205: "ATENAS",
    206: "NARANJO",
    207: "PALMARES",
    208: "POAS",
    209: "OROTINA",
    210: "SAN CARLOS",
    211: "ALFARO RUIZ",
    212: "VALVERDE VEGA",
    213: "UPALA",
    214: "LOS CHILES",
    215: "GUATUSO",
    301: "CARTAGO",
    302: "PARAISO",
    303: "LA UNION",
    304: "JIMENEZ",
    305: "TURRIALBA",
    306: "ALVARADO",
    307: "OREAMUNO",
    308: "EL GUARCO",
    401: "HEREDIA",
    402: "BARVA",
    403: "SANTO DOMINGO",
    404: "SANTA BARBARA",
    405: "SAN RAFAEL",
    406: "SAN ISIDRO",
    407: "BELEN",
    408: "FLORES",
    409: "SAN PABLO",
    410: "SARAPIQUI",
    501: "LIBERIA",
    502: "NICOYA",
    503: "SANTA CRUZ",
    504: "BAGACES",
    505: "CARRILLO",
    506: "CAÑAS",
    507: "ABANGARES",
    508: "TILARAN",
    509: "NANDAYURE",
    510: "LA CRUZ",
    511: "HOJANCHA",
    601: "PUNTARENAS",
    602: "ESPARZA",
    603: "BUENOS AIRES",
    604: "MONTES DE ORO",
    605: "OSA",
    606: "AGUIRRE",
    607: "GOLFITO",
    608: "COTO BRUS",
    609: "PARRITA",
    610: "CORREDORES",
    611: "GARABITO",
    701: "LIMON",
    702: "POCOCI",
    703: "SIQUIRRES",
    704: "TALAMANCA",
    705: "MATINA",
    706: "GUACIMO"
}

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

## Análisis de Series de Tiempo con Clustering

En este caso es de importancia determinar aquellos cantones que tienen una mayor comparación en cómo se comportan, esto a nivel de clima y condiciones que los puede favorecer o perjudicar los casos de dengue.

In [ ]:
df['periodo'] = df['year'].astype(str) + '_' + df['week'].astype(str).str.zfill(2)

In [ ]:
df_pivot = df.pivot_table(index='canton', columns='periodo', values='casos', aggfunc='sum').fillna(0)

In [ ]:
df_pivot = df.pivot_table(index='canton', columns='periodo', values='casos_dengue', aggfunc='sum').fillna(0)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
df_pivot.head()

In [ ]:
df_pivot_scaled = pd.DataFrame(
    scaler.fit_transform(df_pivot.T).T, # Trasponemos para escalar por fila y luego regresamos
    index=df_pivot.index, 
    columns=df_pivot.columns
)

In [ ]:
from tslearn.clustering import TimeSeriesKMeans

In [ ]:
import numpy as np

Determinación del número óptimo de clúster, mediante la técnica del codo, para determinar el número óptimo de grupos que
se pueden ver. En la siguiente figura se observa que hay un total de 4 a 5 grupos, pero nos vamos a guiar por una conceptualización
básica, la cual es tomar el menor número de grupos que se pueda, por tal razón, se toman solo 4 grupos.

In [ ]:
inercias = []
K_rango = range(2, 9)

# Forzamos a usar la misma métrica que usaste (ej. 'dtw' o 'euclidean')
for k in K_rango:
    km = TimeSeriesKMeans(n_clusters=k, metric="dtw", random_state=42, n_jobs=-1)
    km.fit(X_ts) # Tu matriz de 3 dimensiones
    inercias.append(km.inertia_)

# Graficar el Codo
plt.figure(figsize=(8, 5))
plt.plot(K_rango, inercias, marker='o', color='#2b7bba', linestyle='--')
plt.xlabel('Número de Clústeres (K)')
plt.ylabel('Inercia (Suma de Distancias DTW)')
plt.title('Método del Codo para Series Temporales de Dengue')
plt.grid(True, alpha=0.3)
plt.show()

En la comparación del número óptimo de grupos, se va a tomar en cuenta el promedio de Silueta. Aquí se debe
de realizar una distinción clara, los valores van de -1 a 1, aquí se espera que los diferentes cantones
se empiecen a agrupar de manera de buena manera teniendo valores cercanos a 1. Sin embargo, se debe de tomar
en cuenta que este método muestra que realmente no hay grupos, los valores silueta son muy bajos.

Una recomendación, tomando en cuenta que los análisis llevado a cabo y por la complejidad del tema, se recomienda
tomar un número de 4 clústers.

In [ ]:
from tslearn.utils import to_time_series_dataset
from sklearn.metrics import silhouette_score

In [ ]:
for k in K_rango:
    km = TimeSeriesKMeans(n_clusters=k, metric="dtw", random_state=42, n_jobs=-1)
    etiquetas = km.fit_predict(X_ts)
    
    # Reducimos dimensiones para scikit-learn
    X_plana = df_pivot_scaled.drop(columns='cluster', errors='ignore').values
    score = silhouette_score(X_plana, etiquetas)
    print(f"Para K={k}, el Silhouette Score es: {score:.4f}")

## Análisis de cantones registrados mediante clustering de series de tiempo

In [ ]:
n_clusters = 4

model_ts = TimeSeriesKMeans(
    n_clusters=n_clusters, 
    metric="dtw",          # Cambiar a "softdtw" o "euclidean" si el dataset es grande
    max_iter=10, 
    random_state=42,
    n_jobs=-1
)

X_ts = np.expand_dims(df_pivot_scaled.values, axis=-1)
cluster_labels = model_ts.fit_predict(X_ts)

# Añadir los resultados al dataframe original pivotado
df_pivot['cluster'] = cluster_labels
df_pivot_scaled['cluster'] = cluster_labels

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(n_clusters, 1, figsize=(15, 4 * n_clusters), sharex=True)

periodos = df_pivot_scaled.drop(columns='cluster').columns

for i in range(n_clusters):
    df_c = df_pivot_scaled[df_pivot_scaled['cluster'] == i].drop(columns='cluster')
    cantones_en_cluster = df_pivot[df_pivot['cluster'] == i].index.tolist()
    
    for canton in df_c.index:
        axes[i].plot(periodos, df_c.loc[canton], color='gray', alpha=0.15, linewidth=1)
    
    centroide = model_ts.cluster_centers_[i].ravel()
    axes[i].plot(periodos, centroide, color='#e41a1c', linewidth=2.5, label=f'Centroide Clúster {i}')
    
    axes[i].set_title(f'Clúster {i} ({len(df_c)} cantones) | Ejemplo: {[canton_map[i] for i in cantones_en_cluster[:4]]}...', fontsize=12)
    axes[i].set_ylabel('Casos (Escalados)')
    axes[i].grid(True, alpha=0.3)
    
    axes[i].set_xticks(range(0, len(periodos), max(1, len(periodos)//15)))
    axes[i].tick_params(axis='x', rotation=45)

plt.xlabel('Línea del Tiempo (Año_Semana)')
plt.suptitle('Análisis de Clustering de Series Temporales: Patrones Dinámicos del Dengue en CR', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()